In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import math
import random
import seaborn as sns
import matplotlib.pyplot as plt
# Sirve para medir la velocidad de una función
import time
%load_ext line_profiler

In [2]:
# Leemos los datos
df_act_yuc=pd.read_excel(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases/Bases_conteos_yucatan/Casilla_por_casilla_gubernatura_280627_YUC.xlsx")
df_act_yuc

,ID ESTADO,NOMBRE ESTADO,CABECERA DISTRITAL LOCAL,ID MUNICIPIO,DISTRITO,MUNICIPIO,SECCION,CASILLA,TIPO DE CASILLA,ID CASILLA,...,PVEM-PT-MORENA,PVEM-PT,PVEM-MORENA,PT-MORENA,CANDIDATOS NO REGISTRADOS,VOTOS NULOS,LISTA NOMINAL,TOTAL,PARTICIPACIÓN,Unnamed: 34
0,31,YUCATÁN,UMAN,1,12,ABALA,1,'0001B1,B,1,...,2.0,1.0,0.0,0.0,0.0,12.0,596.0,532.0,0.892617,NaN
1,31,YUCATÁN,UMAN,1,12,ABALA,1,'0001C1,C,2,...,1.0,0.0,2.0,0.0,0.0,11.0,596.0,547.0,0.917785,NaN
2,31,YUCATÁN,UMAN,1,12,ABALA,1,'0001C2,C,3,...,5.0,0.0,2.0,0.0,0.0,13.0,595.0,542.0,0.910924,NaN
3,31,YUCATÁN,UMAN,1,12,ABALA,2,'0002B1,B,4,...,0.0,0.0,0.0,1.0,0.0,3.0,174.0,162.0,0.931034,NaN
4,31,YUCATÁN,UMAN,1,12,ABALA,3,0003B1,B,5,...,NaN,NaN,NaN,NaN,NaN,NaN,205.0,NaN,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2974,31,YUCATÁN,MOTUL,54,15,MUXUPIP,1203,1203V1,VA,2975,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2975,31,YUCATÁN,MOTUL,93,15,TIXKOKOB,1204,1204V1,VA,2976,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2976,31,YUCATÁN,TIZIMIN,96,17,SAN FELIPE,1205,1205V1,VA,2977,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2977,31,YUCATÁN,TIZIMIN,96,17,TIZIMIN,1206,1206V1,VA,2978,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Limpieza de datos

In [3]:
# Hay que revisar si se quitan los de SIN RESULTADO, por ahora solo quitaré los PENDIENTE DE CAPTURA
#df_act_yuc_r=df_act_yuc.copy()
df_act_yuc_r=df_act_yuc[(~(df_act_yuc["ESTATUS ACTA"].isin(["PENDIENTE DE CAPTURA","SIN RESULTADO"]))) & (df_act_yuc["TOTAL"]>0)].reset_index(drop=True)
# Quitamos la última columna
df_act_yuc_r=df_act_yuc_r.drop(df_act_yuc_r.columns[len(df_act_yuc_r.columns)-1], axis=1)
# Llenamos los missing con cero de los votos y totales
df_act_yuc_r.iloc[:,13:]=df_act_yuc_r.iloc[:,13:].fillna(0)
df_act_yuc_r

,ID ESTADO,NOMBRE ESTADO,CABECERA DISTRITAL LOCAL,ID MUNICIPIO,DISTRITO,MUNICIPIO,SECCION,CASILLA,TIPO DE CASILLA,ID CASILLA,...,PRI-NAY,PVEM-PT-MORENA,PVEM-PT,PVEM-MORENA,PT-MORENA,CANDIDATOS NO REGISTRADOS,VOTOS NULOS,LISTA NOMINAL,TOTAL,PARTICIPACIÓN
0,31,YUCATÁN,UMAN,1,12,ABALA,1,'0001B1,B,1,...,0.0,2.0,1.0,0.0,0.0,0.0,12.0,596.0,532.0,0.892617
1,31,YUCATÁN,UMAN,1,12,ABALA,1,'0001C1,C,2,...,0.0,1.0,0.0,2.0,0.0,0.0,11.0,596.0,547.0,0.917785
2,31,YUCATÁN,UMAN,1,12,ABALA,1,'0001C2,C,3,...,0.0,5.0,0.0,2.0,0.0,0.0,13.0,595.0,542.0,0.910924
3,31,YUCATÁN,UMAN,1,12,ABALA,2,'0002B1,B,4,...,0.0,0.0,0.0,0.0,1.0,0.0,3.0,174.0,162.0,0.931034
4,31,YUCATÁN,UMAN,1,12,ABALA,4,'0004B1,B,6,...,0.0,4.0,0.0,0.0,0.0,0.0,14.0,550.0,513.0,0.932727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2836,31,YUCATÁN,MERIDA,50,3,MERIDA,1195,'1195V1,VA,2967,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.000000
2837,31,YUCATÁN,MERIDA,50,5,MERIDA,1197,'1197V1,VA,2969,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.000000
2838,31,YUCATÁN,MERIDA,50,8,MERIDA,1199,'1199V1,VA,2971,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.000000
2839,31,YUCATÁN,MERIDA,50,9,MERIDA,1200,'1200V1,VA,2972,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.000000


In [4]:
import warnings
warnings.filterwarnings("ignore")
# Nos quedamos con las columnas necesarias
df_act_yuc_i=df_act_yuc_r[["ID ESTADO","NOMBRE ESTADO","DISTRITO","SECCION","CASILLA","ID CASILLA","ESTATUS ACTA",'PAN', 'PRI', 'PRD', 'PVEM', 'PT', 'MC', 'MORENA',
        'NAY', 'PAN-PRI-NAY', 'PAN-PRI', 'PAN-NAY', 'PRI-NAY', 'PVEM-PT-MORENA',
       'PVEM-PT', 'PVEM-MORENA', 'PT-MORENA', 'CANDIDATOS NO REGISTRADOS',
       'VOTOS NULOS', 'LISTA NOMINAL', 'TOTAL', 'PARTICIPACIÓN']]
# Creamos los votos por coalición
# Candidato sigamos haciendo historia 
df_act_yuc_i["JOAQUIN_DIAZ_MENA"]=df_act_yuc_i.loc[:,"MORENA"]+df_act_yuc_i.loc[:,"PT"]+df_act_yuc_i.loc[:,"PVEM"]+df_act_yuc_i.loc[:,"PVEM-MORENA"]+df_act_yuc_i.loc[:,"PVEM-PT"]+df_act_yuc_i.loc[:,"PT-MORENA"]+df_act_yuc_i.loc[:,"PVEM-PT-MORENA"]
# Candidato Unidos gana Yucatán
df_act_yuc_i["RENAN_BARRERA_CONCHA"]=df_act_yuc_i.loc[:,"PAN"]+df_act_yuc_i.loc[:,"PRI"]+df_act_yuc_i.loc[:,"NAY"]+df_act_yuc_i.loc[:,"PAN-PRI"]+df_act_yuc_i.loc[:,"PAN-NAY"]+df_act_yuc_i.loc[:,"PRI-NAY"]+df_act_yuc_i.loc[:,"PAN-PRI-NAY"]
# Candidato MC
df_act_yuc_i["VIDA_ARAVARI_GOMEZ_HERRERA"]=df_act_yuc_i.loc[:,"MC"]
# Canidato PRD
df_act_yuc_i["YAMIL_JASMIN_LOPEZ_MANRIQUE"]=df_act_yuc_i.loc[:,"PRD"]
# Votos nulos y candidatos no registrados
df_act_yuc_i["VOTOS_NULOS_CAND_NO_REGIS"]=df_act_yuc_i.loc[:,"VOTOS NULOS"]+df_act_yuc_i.loc[:,"CANDIDATOS NO REGISTRADOS"]
# Datos reales de la proporción
df_act_yuc_i

,ID ESTADO,NOMBRE ESTADO,DISTRITO,SECCION,CASILLA,ID CASILLA,ESTATUS ACTA,PAN,PRI,PRD,...,CANDIDATOS NO REGISTRADOS,VOTOS NULOS,LISTA NOMINAL,TOTAL,PARTICIPACIÓN,JOAQUIN_DIAZ_MENA,RENAN_BARRERA_CONCHA,VIDA_ARAVARI_GOMEZ_HERRERA,YAMIL_JASMIN_LOPEZ_MANRIQUE,VOTOS_NULOS_CAND_NO_REGIS
0,31,YUCATÁN,12,1,'0001B1,1,COMPUTADA,171.0,107.0,3.0,...,0.0,12.0,596.0,532.0,0.892617,209.0,290.0,18.0,3.0,12.0
1,31,YUCATÁN,12,1,'0001C1,2,COMPUTADA,177.0,104.0,0.0,...,0.0,11.0,596.0,547.0,0.917785,208.0,299.0,29.0,0.0,11.0
2,31,YUCATÁN,12,1,'0001C2,3,COMPUTADA,176.0,100.0,1.0,...,0.0,13.0,595.0,542.0,0.910924,228.0,289.0,11.0,1.0,13.0
3,31,YUCATÁN,12,2,'0002B1,4,COMPUTADA,46.0,33.0,2.0,...,0.0,3.0,174.0,162.0,0.931034,71.0,79.0,7.0,2.0,3.0
4,31,YUCATÁN,12,4,'0004B1,6,COMPUTADA,110.0,96.0,7.0,...,0.0,14.0,550.0,513.0,0.932727,275.0,207.0,10.0,7.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2836,31,YUCATÁN,3,1195,'1195V1,2967,COMPUTADA,2.0,0.0,0.0,...,0.0,0.0,0.0,2.0,0.000000,0.0,2.0,0.0,0.0,0.0
2837,31,YUCATÁN,5,1197,'1197V1,2969,COMPUTADA,2.0,1.0,0.0,...,0.0,0.0,0.0,5.0,0.000000,2.0,3.0,0.0,0.0,0.0
2838,31,YUCATÁN,8,1199,'1199V1,2971,COMPUTADA,2.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.000000,1.0,2.0,0.0,0.0,0.0
2839,31,YUCATÁN,9,1200,'1200V1,2972,COMPUTADA,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.000000,1.0,0.0,0.0,0.0,0.0


In [5]:
# Datos reales de la proporción
prop_reales = df_act_yuc_i[["JOAQUIN_DIAZ_MENA", "RENAN_BARRERA_CONCHA", "VIDA_ARAVARI_GOMEZ_HERRERA", "YAMIL_JASMIN_LOPEZ_MANRIQUE", "VOTOS_NULOS_CAND_NO_REGIS"]].sum() / df_act_yuc_i[["JOAQUIN_DIAZ_MENA", "RENAN_BARRERA_CONCHA", "VIDA_ARAVARI_GOMEZ_HERRERA", "YAMIL_JASMIN_LOPEZ_MANRIQUE", "VOTOS_NULOS_CAND_NO_REGIS"]].sum().sum()
prop_reales

JOAQUIN_DIAZ_MENA              0.515125
RENAN_BARRERA_CONCHA           0.421366
VIDA_ARAVARI_GOMEZ_HERRERA     0.036792
YAMIL_JASMIN_LOPEZ_MANRIQUE    0.005249
VOTOS_NULOS_CAND_NO_REGIS      0.021467
dtype: float64

In [ ]:
# Guardamos la base con la que vamos a hacer el análisis
#df_act_yuc_i.to_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases/Base elecciones analsis/base_elecciones_yuc_analsis.csv")